In [0]:
#Advanced
# Focus: Conform addresses with geo enrichment; build address dimension and bridge.
# Uses: dfAddress, dfAddressType, dfBusinessEntity, dfBusinessEntityAddress, dfStateProvince, dfCountryRegion
# Writes:

# …/dim_address/ (Type‑1, conformed)
# …/person_address/ (bridge with role)
# …/person_canonical_address/ (one “best” address per person, by role priority)
# …/dim_address_quarantine/ (DQ rejects; optional)
# …/_run_summaries/address/ (run metrics)


# Note: No SQL, no Delta, only Parquet. Deterministic keys via SHA‑256 over natural keys.

In [0]:
# =========================
# DEMO PROLOGUE (safe defaults)
# =========================
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")
from pyspark.sql import functions as F, Window

# ---- Runtime toggles (demo-safe defaults) ----
demo_mode = True                    # If True, relaxes strict DQ; still writes useful outputs.
enable_incremental = False          # If True, requires you to set 'watermark_from' below.
allow_late_arrival = False          # Not used in this notebook; present for symmetry.
enable_pit_snapshot = False         # Not used in this notebook; present for symmetry.

# Role priority for canonical selection (edit if desired)
role_priority = ["Home", "Main Office", "Billing", "Shipping"]

# ---- Preferred output base (ABFS/OneLake), with fallbacks for demos ----
silver_base_path = "abfss://fabmigation1@synpasetofabric.dfs.core.windows.net/silver/adventureworks/"
fallback_path = "/synapse/workdir/silver"
fallback_alt = "/tmp/silver"


bronze_path = f"/Volumes/main/default/fabmigration1_bronze/adventureworks/{schema_name}.{table_name}.parquet/"
base_silver_root = f"main.silver.{schema_name}_{table_name}" 

In [0]:
def _can_write(path: str) -> bool:
    try:
        test_df = spark.createDataFrame([(1,)], ["x"])
        test_tmp = path.rstrip("/") + "/_write_probe__"
        test_df.write.mode("overwrite").parquet(test_tmp)
        # best-effort cleanup
        try:
            spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration()) \
                .delete(spark._jvm.org.apache.hadoop.fs.Path(test_tmp), True)
        except Exception:
            pass
        return True
    except Exception:
        return False
if _can_write(silver_base_path):
    out_base = silver_base_path
else:
    out_base = silver_base_path
print(f"[INFO] Using output base: {out_base}")

In [0]:
def parquet_exists(path: str) -> bool:
    try:
        spark.read.parquet(path).limit(1).count()
        return True
    except Exception:
        return False

def read_parquet_if_exists(path: str):
    return spark.read.parquet(path) if parquet_exists(path) else None


In [0]:
print(out_base)

In [0]:

# =========================
# OUTPUT PATHS
# =========================
out_dim_address            = f"{out_base}/dim_address"
out_person_address         = f"{out_base}/person_address"
out_person_canonical_addr  = f"{out_base}/person_canonical_address"
out_addr_quarantine        = f"{out_base}/dim_address_quarantine"
out_summary_addr           = f"{out_base}/_run_summaries/address"


In [0]:
base = "abfss://fabmigation1@synpasetofabric.dfs.core.windows.net/bronze/adventureworks/Person"

tables = ["Address","AddressType","BusinessEntity","BusinessEntityAddress","StateProvince","CountryRegion"]

dfs = {t: spark.read.parquet(f"{base}.{t}.parquet/") for t in tables}

dfAddress = dfs["Address"]; dfAddressType = dfs["AddressType"]; dfBusinessEntity = dfs["BusinessEntity"]
dfBusinessEntityAddress = dfs["BusinessEntityAddress"]; dfStateProvince = dfs["StateProvince"]; dfCountryRegion = dfs["CountryRegion"]
display(dfAddress)

In [0]:
# =========================
# 1) Conform & build dim_address (Type-1)
# =========================

# Geo lookups
sp = dfStateProvince.select("StateProvinceID", "StateProvinceCode", "CountryRegionCode", "Name") \
                    .withColumnRenamed("Name", "StateProvinceName")
cr = dfCountryRegion.select("CountryRegionCode", "Name").withColumnRenamed("Name", "CountryName")

# Optional incremental filter
addr_src = dfAddress
if enable_incremental and watermark_from:
    addr_src = addr_src.filter(F.col("ModifiedDate") >= F.to_timestamp(F.lit(watermark_from)))

# Conform and enrich
addr = (addr_src.alias("a")
        .join(sp.alias("sp"), "StateProvinceID", "left")
        .join(cr.alias("cr"), "CountryRegionCode", "left")
        .select(
            "a.AddressID",
            F.upper(F.trim(F.col("a.AddressLine1"))).alias("AddressLine1"),
            F.upper(F.trim(F.col("a.AddressLine2"))).alias("AddressLine2"),
            F.upper(F.trim(F.col("a.City"))).alias("City"),
            F.upper(F.trim(F.col("PostalCode"))).alias("PostalCode"),
            F.upper(F.trim(F.col("sp.StateProvinceCode"))).alias("StateProvinceCode"),
            F.col("sp.StateProvinceName"),
            F.upper(F.trim(F.col("cr.CountryRegionCode"))).alias("CountryRegionCode"),
            F.col("cr.CountryName"),
            "a.rowguid", "a.ModifiedDate"
        )
)

# Natural key across normalized fields
addr = addr.withColumn(
    "address_nk",
    F.sha2(F.concat_ws("||",
        F.coalesce(F.col("AddressLine1"), F.lit("")),
        F.coalesce(F.col("AddressLine2"), F.lit("")),
        F.coalesce(F.col("City"), F.lit("")),
        F.coalesce(F.col("StateProvinceCode"), F.lit("")),
        F.coalesce(F.col("PostalCode"), F.lit("")),
        F.coalesce(F.col("CountryRegionCode"), F.lit(""))
    ), 256)
)

# Simple DQ: missing country/state; postal validation for US/CA/GB (relaxed in demo)
enable_dq = True
us_re   = r"^\d{5}(-\d{4})?$"
ca_re   = r"^[ABCEGHJ-NPRSTVXY]\d[ABCEGHJ-NPRSTV-Z][ ]?\d[ABCEGHJ-NPRSTV-Z]\d$"
gb_re   = r"^[A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2}$"

def postal_valid_expr():
    return (
        F.when(F.col("CountryRegionCode") == "US", F.col("PostalCode").rlike(us_re))
         .when(F.col("CountryRegionCode") == "CA", F.regexp_replace(F.col("PostalCode"), " ", "").rlike(ca_re))
         .when(F.col("CountryRegionCode") == "GB", F.regexp_replace(F.col("PostalCode"), " ", "").rlike(gb_re))
         .otherwise(F.lit(True if demo_mode else False))
    )

dq_conditions = [
    F.col("CountryRegionCode").isNotNull(),
    F.col("StateProvinceCode").isNotNull(),
    postal_valid_expr()
]

if enable_dq:
    addr_valid   = addr.where(dq_conditions[0] & dq_conditions[1] & dq_conditions[2])
    addr_rejects = (addr.withColumn(
        "reject_reason",
        F.when(F.col("CountryRegionCode").isNull(), F.lit("MISSING_COUNTRY"))
         .when(F.col("StateProvinceCode").isNull(), F.lit("MISSING_STATEPROVINCE"))
         .when(~postal_valid_expr(), F.lit("POSTAL_INVALID"))
         .otherwise(F.lit("UNKNOWN"))
    ).where(F.col("reject_reason").isNotNull()))
else:
    addr_valid   = addr
    addr_rejects = spark.createDataFrame([], schema=addr.limit(0).schema.add("reject_reason", "string"))

# Dedupe by NK (keep latest ModifiedDate)
w_addr = Window.partitionBy("address_nk").orderBy(F.col("ModifiedDate").desc_nulls_last())
dim_address_df = (
    addr_valid.withColumn("rn", F.row_number().over(w_addr))
              .filter("rn = 1")
              .drop("rn")
              .withColumn("address_sk", F.sha2(F.col("address_nk"), 256))
              .withColumn("ingestion_timestamp", F.current_timestamp())
              .select(
                  "address_sk", "address_nk", "AddressID", "AddressLine1", "AddressLine2", "City",
                  "StateProvinceCode", "StateProvinceName", "PostalCode",
                  "CountryRegionCode", "CountryName", "ModifiedDate", "ingestion_timestamp"
              )
)

# Optional: detect NK "collisions" (should be zero)
collisions = (dim_address_df.groupBy("address_nk").count().where("count > 1"))

# Write outputs
dim_address_df.write.mode("overwrite").parquet(out_dim_address)
if addr_rejects.count() > 0:
    addr_rejects.write.mode("overwrite").parquet(out_addr_quarantine)

# Run summary for dim_address
existing_dim = read_parquet_if_exists(out_dim_address)  # after write = current snapshot
prev_dim     = read_parquet_if_exists(f"{out_dim_address}_prev")  # look for prev if kept; not required

summary_dim = spark.createDataFrame([(
    int(addr.count()),                           # total_input_rows
    int(addr_valid.count()),                     # valid_rows
    int(addr_rejects.count()),                   # quarantined_rows
    int(dim_address_df.select("address_nk").distinct().count()),  # distinct_nk
    int(collisions.count())                      # nk_collisions
)], ["total_input_rows","valid_rows","quarantined_rows","distinct_nk","nk_collisions"]) \
.withColumn("entity", F.lit("dim_address")) \
.withColumn("run_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss")) \
.withColumn("run_ts", F.current_timestamp())

summary_dim.write.mode("append").parquet(out_summary_addr)


In [0]:
display(dim_address_df.limit(50)); display(addr_rejects.limit(50)); display(summary_dim)

In [0]:
# =========================
# 2) Person–Address bridge (via BusinessEntityAddress + AddressType)
# =========================

# Limit to valid BusinessEntityID
bea_valid = (dfBusinessEntityAddress.alias("bea")
             .join(dfBusinessEntity.select("BusinessEntityID").alias("be"), "BusinessEntityID", "inner"))

addr_type = dfAddressType.select("AddressTypeID", F.col("Name").alias("address_role"))

bridge_raw = (bea_valid
              .join(addr_type, "AddressTypeID", "left")
              .select("BusinessEntityID", "AddressID", "address_role", "rowguid", "ModifiedDate"))

# Map AddressID -> SK using in-memory dim
dim_lookup = dim_address_df.select("AddressID","address_sk","address_nk")
bridge = (bridge_raw.join(dim_lookup, "AddressID", "left")
          .withColumn("bridge_nk", F.sha2(F.concat_ws("||",
                                                      F.col("BusinessEntityID").cast("string"),
                                                      F.coalesce(F.col("address_sk"), F.lit("")),
                                                      F.coalesce(F.col("address_role"), F.lit(""))), 256))
)

# Dedupe bridge by NK (latest ModifiedDate)
w_bridge = Window.partitionBy("bridge_nk").orderBy(F.col("ModifiedDate").desc_nulls_last())
person_address = (
    bridge.withColumn("rn", F.row_number().over(w_bridge))
          .filter("rn = 1")
          .drop("rn")
          .withColumn("person_address_sk", F.col("bridge_nk"))
          .withColumn("ingestion_timestamp", F.current_timestamp())
          .select("person_address_sk","BusinessEntityID","address_sk","address_role","ModifiedDate","ingestion_timestamp")
)

person_address.write.mode("overwrite").parquet(out_person_address)

# Run summary for bridge
summary_bridge = spark.createDataFrame([(
    int(bridge_raw.count()),
    int(person_address.count()),
    int(person_address.select("BusinessEntityID").distinct().count())
)], ["raw_links","curated_links","distinct_persons"]) \
.withColumn("entity", F.lit("person_address")) \
.withColumn("run_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss")) \
.withColumn("run_ts", F.current_timestamp())

summary_bridge.write.mode("append").parquet(out_summary_addr)
display(summary_bridge)

In [0]:
display(person_address.limit(50))

In [0]:
# =========================
# 3) Canonical address per person (role priority + recency)
# =========================

# Role ranking (lower is better)
role_rank_expr = F.when(F.col("address_role").isNull(), F.lit(9999))
for idx, role in enumerate(role_priority):
    role_rank_expr = F.when(F.col("address_role") == role, F.lit(idx)).otherwise(role_rank_expr)

canon_w = Window.partitionBy("BusinessEntityID") \
                .orderBy(role_rank_expr.asc(), F.col("ModifiedDate").desc_nulls_last(), F.col("person_address_sk").asc())

person_canonical_address = (
    person_address
    .withColumn("role_rank", role_rank_expr)
    .withColumn("rn", F.row_number().over(canon_w))
    .filter("rn = 1")
    .drop("rn")
    .select(
        "BusinessEntityID",
        F.col("address_sk").alias("canonical_address_sk"),
        F.col("address_role").alias("canonical_role"),
        "ModifiedDate"
    )
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

person_canonical_address.write.mode("overwrite").parquet(out_person_canonical_addr)

# Run summary for canonical
summary_canon = spark.createDataFrame([(
    int(person_canonical_address.count()),
    ", ".join(role_priority)
)], ["canonical_rows","role_priority_order"]) \
.withColumn("entity", F.lit("person_canonical_address")) \
.withColumn("run_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss")) \
.withColumn("run_ts", F.current_timestamp())

summary_canon.write.mode("append").parquet(out_summary_addr)

print("[OK] Notebook 3 completed.")

In [0]:
display(person_canonical_address.limit(50)); display(summary_canon)